# M3 — MemoryS7-CueStart staged frozen memory interface

Stage A 从冻结的 memory-bearing world model 蒸馏十原子头；Stage B 冻结 world model 和 head，只训练 atoms-only actor/critic，并用 400 条 privileged scripted demonstrations 辅助覆盖。

**论文最终结论是负面的语义门控结果：** 两个 actor 都有 237/500 baseline-correct episodes；cue-mid flip 对这些 episodes 产生 **0/237 target→other**、**151/237 target→timeout**。因此接口 readable 且 policy-sensitive，但没有证明可编辑语义控制。

In [1]:
import json, pathlib, sys, warnings
warnings.filterwarnings("ignore")
import numpy as np
import torch
import matplotlib
matplotlib.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DengXian"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

HERE = pathlib.Path.cwd()
REPO = HERE if (HERE / "dreamer.py").exists() else HERE.parent
sys.path[:0] = [str(REPO), str(REPO / "scripts")]
import envs.minigrid as M
import staged_stage_c_eval as C
from evaluate_m3_corrected_flip import outcome

MODEL_DIR = REPO / "models/memory/memory_experiment_3_m3"
RESULT_DIR = REPO / "results/memory_experiment_3_m3"
CHECKPOINT = MODEL_DIR / "actor_seed0_400demo.pt"
META = MODEL_DIR / "frozen_symbolic_head_meta.json"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
seed0_result = json.loads((RESULT_DIR / "m3_seed0_corrected_500.json").read_text(encoding="utf-8"))
seed1_result = json.loads((RESULT_DIR / "m3_seed1_corrected_500.json").read_text(encoding="utf-8"))
agent, config = C.load(str(CHECKPOINT), DEVICE, distill_meta_path=str(META), logdir=MODEL_DIR)
wm, behavior = agent._wm, agent._task_behavior
assert wm._sym_policy_input == "atoms"
print("device:", DEVICE)
print("actor input:", behavior.actor.layers[0].in_features, "named atoms")
print("labels:", list(wm._sym_head.labels))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Encoder CNN shapes: {}
Encoder MLP shapes: {'grid': (148,)}
Decoder CNN shapes: {}
Decoder MLP shapes: {'grid': (148,)}


Symbolic belief head enabled: feat1280->10 atoms labels=['cue_is_key', 'wall_ahead', 'key_ahead', 'key_left', 'key_right', 'key_reach', 'ball_ahead', 'ball_left', 'ball_right', 'ball_reach'] n_lit=32 mask=cue_known loss=mse scale=2.0 (atoms shape WM AND feed policy)


Optimizer model_opt has 12498180 variables.
Optimizer actor_opt has 70405 variables.
Optimizer value_opt has 134655 variables.
[load] 旧checkpoint兼容：手动恢复 _wm._sym_head.anneal_steps = 4800
[load] 旧checkpoint兼容：手动恢复 _task_behavior._world_model._sym_head.anneal_steps = 4800
[load] 旧checkpoint兼容：手动恢复 _expl_behavior._world_model._sym_head.anneal_steps = 4800
[load] C:\Users\super\Desktop\test\Xdreamer\models\memory\memory_experiment_3_m3\actor_seed0_400demo.pt | missing 0 unexpected 0
device: cuda:0
actor input: 10 named atoms
labels: ['cue_is_key', 'wall_ahead', 'key_ahead', 'key_left', 'key_right', 'key_reach', 'ball_ahead', 'ball_left', 'ball_right', 'ball_reach']


## A. 冻结的论文证据（两个 actor）

In [2]:
for actor_seed, result in [(0, seed0_result), (1, seed1_result)]:
    primary = result["comparisons"]["cue_mid"]
    wall = result["comparisons"]["wall_mid"]
    transitions = primary["transitions"]
    print(f"actor seed {actor_seed}")
    print(" baseline:", primary["base"])
    print(" cue-mid target->other:", transitions["target_to_other"], "/", primary["baseline_target_n"])
    print(" cue-mid target->timeout:", transitions["target_to_timeout"], "/", primary["baseline_target_n"])
    print(" wall-mid target->timeout:", wall["transitions"]["target_to_timeout"], "/", wall["baseline_target_n"])
    print(" semantic support gate:", result["predeclared_support_gate"]["passed"], "\n")

actor seed 0
 baseline: {'target': 237, 'other': 140, 'timeout': 123, 'neither': 0}
 cue-mid target->other: 0 / 237
 cue-mid target->timeout: 151 / 237
 wall-mid target->timeout: 237 / 237
 semantic support gate: False 

actor seed 1
 baseline: {'target': 237, 'other': 263, 'timeout': 0, 'neither': 0}
 cue-mid target->other: 0 / 237
 cue-mid target->timeout: 151 / 237
 wall-mid target->timeout: 237 / 237
 semantic support gate: False 



## B. 同一 seed 的 baseline 与 cue-mid flip 可视化

下面使用 corrected JSON 中的 seed 1000：baseline 到正确目标；从 step 3 开始持续翻转 `cue_is_key` 后超时。动画只为人类核验轨迹，正式统计仍来自两份 500-episode JSON。

In [3]:
@torch.no_grad()
def visual_rollout(seed, flip_start=None):
    env = M.MiniGrid("memoryS7_cuestart", mode="eval", seed=seed, max_steps=config.time_limit, render_obs=False, emit_labels=False)
    obs = env.reset()
    latent = action = None
    frames, actions, cue_atoms = [], [], []
    info = {}
    success = False
    for step in range(config.time_limit):
        batch = {k: np.asarray(v)[None] for k, v in obs.items() if not k.startswith("log_")}
        data = wm.preprocess(batch)
        embed = wm.encoder(data)
        latent, _ = wm.dynamics.obs_step(latent, action, embed, data["is_first"], sample=False)
        feat = wm.dynamics.get_feat(latent)
        atoms = wm._sym_head.atoms(feat).detach()
        policy_atoms = atoms.clone()
        if flip_start is not None and step >= flip_start:
            policy_atoms[0, 0] = -policy_atoms[0, 0]
        action = behavior.actor(policy_atoms).mode()
        action_index = int(torch.argmax(action, dim=-1)[0].item())
        frames.append(env.render())
        actions.append(action_index)
        cue_atoms.append(float(policy_atoms[0, 0].item()))
        obs, reward, done, info = env.step(action_index)
        if done:
            success = bool(reward > 0)
            break
    frames.append(env.render())
    final = env.god_state()
    result = {
        "success": success,
        "steps": step + 1,
        "final_pos": final["agent_pos"],
        "key_pos": final["key_pos"],
        "ball_pos": final["ball_pos"],
        "cue_is_key": final["cue_is_key"],
        "truncated": "discount" in info,
    }
    env.close()
    return {"frames": frames, "actions": actions, "cue_atoms": cue_atoms, "result": result}

record = next(x for x in seed0_result["records"] if x["seed"] == 1000)
flip_step = record["flip_steps"]["mid"]
baseline_demo = visual_rollout(1000)
flipped_demo = visual_rollout(1000, flip_start=flip_step)
print("JSON record:", record["baseline"], "->", record["cue_mid"], "flip step", flip_step)
print("live replay:", outcome(baseline_demo["result"]), "->", outcome(flipped_demo["result"]))

JSON record: {'outcome': 'target', 'steps': 6} -> {'outcome': 'timeout', 'steps': 250} flip step 3
live replay: target -> timeout


In [4]:
ACTION_NAMES = ["left", "right", "forward", "pickup", "toggle"]
max_steps = max(len(baseline_demo["frames"]), len(flipped_demo["frames"]))
shown = sorted(set(list(range(min(15, max_steps))) + np.linspace(15, max_steps - 1, 35, dtype=int).tolist()))
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.8))
images = [axes[0].imshow(baseline_demo["frames"][0]), axes[1].imshow(flipped_demo["frames"][0])]
for ax in axes:
    ax.axis("off")

def update(frame_number):
    step = shown[frame_number]
    for index, (demo, title) in enumerate([(baseline_demo, "baseline"), (flipped_demo, "cue-mid flip")]):
        j = min(step, len(demo["frames"]) - 1)
        images[index].set_data(demo["frames"][j])
        cue = demo["cue_atoms"][min(j, len(demo["cue_atoms"]) - 1)] if demo["cue_atoms"] else float("nan")
        axes[index].set_title(f"{title} | step={j} | policy cue atom={cue:+.2f}")
    return images

anim = animation.FuncAnimation(fig, update, frames=len(shown), interval=450, blit=False)
html = HTML(anim.to_jshtml())
plt.close(fig)
display(html)

## 验收结论

- M3 的十原子 frozen head 是 readable，actor/critic 没有 continuous bypass。
- 两个 actor 的 baseline-correct 都是 237/500，但它们共享 world model 和 head，因此不是独立 representation replications。
- cue-mid flip 的严格语义结果是 **0/237 target→other**；151/237 变成 timeout。
- wall-mid flip 使 237/237 baseline-correct episodes timeout，说明“产生破坏”不能等同于“按语义可编辑”。
- 支持的表述是 **demonstration-assisted, readable, policy-sensitive frozen memory interface**；不支持“可编辑记忆接口”。